# AITM Red-Teaming — Colab Runner (All-in-Colab)

**Everything runs inside Colab** — vLLM serves `gemma-4-31b-it` on `localhost:8000`,  
and `benchmark.py` calls it directly. No ngrok needed.

**GPU required:** A100 40 GB (Colab Pro) — 4-bit quant  
**Persistence:** uv venv + model weights cached on Google Drive → fast reconnect

---
### First-run checklist
1. Runtime → Change runtime type → **A100 GPU**
2. Set `HF_TOKEN` in Cell 4 (Hugging Face token with Gemma access)
3. Run all cells top-to-bottom

### Reconnect (subsequent sessions)
- Cells 1-4: mount Drive + restore uv/venv (< 1 min, no reinstall)
- Cell 5: start vLLM (loads weights from Drive, ~2-3 min)
- Cell 6+: run experiments

In [ ]:
# ── CELL 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/aitm-env'
!mkdir -p {DRIVE}/hf_cache
print('Drive mounted at', DRIVE)

In [ ]:
# ── CELL 2: Install / restore uv from Drive ───────────────────────────────────
import os, subprocess

UV_CACHED = f'{DRIVE}/uv'
UV_BIN    = '/usr/local/bin/uv'

if os.path.exists(UV_CACHED):
    subprocess.run(f'cp {UV_CACHED} {UV_BIN} && chmod +x {UV_BIN}', shell=True)
    print('uv restored from Drive')
else:
    subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True)
    uv_installed = subprocess.run('find ~/.local/bin /root/.local/bin -name uv 2>/dev/null | head -1',
                                  shell=True, capture_output=True, text=True).stdout.strip()
    subprocess.run(f'cp {uv_installed} {UV_CACHED} && cp {uv_installed} {UV_BIN} && chmod +x {UV_BIN}', shell=True)
    print('uv installed and cached to Drive')

!uv --version

In [ ]:
# ── CELL 3: Create / restore venv on Drive ────────────────────────────────────
VENV   = f'{DRIVE}/.venv'
PYTHON = f'{VENV}/bin/python'

if not os.path.exists(f'{VENV}/bin/python'):
    print('Creating venv on Drive (first time, ~5-10 min)...')
    !uv venv {VENV} --python 3.12
    !uv pip install --python {PYTHON} vllm
    print('venv created and saved to Drive')
else:
    print('venv restored from Drive — skipping install')

!{PYTHON} --version

In [ ]:
# ── CELL 4: Set tokens + HF cache path ───────────────────────────────────────
import os

HF_TOKEN = ''   # huggingface.co/settings/tokens (needs Gemma access)

os.environ['HF_TOKEN']               = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
os.environ['HF_HOME']                = f'{DRIVE}/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE']  = f'{DRIVE}/hf_cache'

# Model weights download here on first run (~60 GB), reused every session after
print('HF cache →', os.environ['HF_HOME'])

In [ ]:
# ── CELL 5: Start vLLM on localhost:8000 ─────────────────────────────────────
# First run: downloads weights to Drive (~60 GB, 15-30 min)
# Subsequent runs: loads from Drive cache (~2-3 min)
import subprocess, time, urllib.request

MODEL = 'google/gemma-4-31b-it'
PORT  = 8000

proc = subprocess.Popen(
    [
        PYTHON, '-m', 'vllm.entrypoints.openai.api_server',
        '--model',                  MODEL,
        '--max-model-len',          '8192',
        '--gpu-memory-utilization', '0.90',
        '--tensor-parallel-size',   '1',
        '--port',                   str(PORT),
    ],
    stdout=open(f'{DRIVE}/vllm.log', 'w'),
    stderr=subprocess.STDOUT,
)
print(f'vLLM PID {proc.pid} starting... (logs → {DRIVE}/vllm.log)')

for i in range(120):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/health')
        print(f'vLLM ready on localhost:{PORT} after {i*5}s')
        break
    except:
        time.sleep(5)
else:
    print('Timeout — check logs:', f'{DRIVE}/vllm.log')

In [ ]:
# ── CELL 6: Clone / update repo ──────────────────────────────────────────────
REPO   = 'https://github.com/highphysicist/aitm-red-teaming-mas.git'
BRANCH = 'Gemma4-colab'
REPO_DIR = '/content/aitm'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}

# Install project dependencies into Drive venv (skips already-installed packages)
!uv pip install --python {PYTHON} -r requirements.txt -q
print('Repo ready at', REPO_DIR)

In [ ]:
# ── CELL 7: Run benchmark ─────────────────────────────────────────────────────
# config.py on Gemma4-colab already points to localhost:8000
# Edit the args below as needed

!{PYTHON} benchmark.py \
    --adapter autogen \
    --topo chain \
    --dataset mbpp \
    --n_samples 20 \
    --attack_type targeted \
    --k 1 \
    --save_log

In [ ]:
# ── CELL 8 (optional): Run with --judge defense ───────────────────────────────
# Judge also uses localhost:8000 (JUDGE_BACKEND="local" in config.py)

!{PYTHON} benchmark.py \
    --adapter autogen \
    --topo chain \
    --dataset mbpp \
    --n_samples 20 \
    --attack_type targeted \
    --k 1 \
    --judge \
    --save_log

In [ ]:
# ── CELL 9 (optional): Copy results to Drive for safekeeping ─────────────────
import shutil, glob

RESULTS_DRIVE = f'{DRIVE}/results'
os.makedirs(RESULTS_DRIVE, exist_ok=True)

for f in glob.glob(f'{REPO_DIR}/results_*.json'):
    shutil.copy(f, RESULTS_DRIVE)
    print('Saved:', os.path.basename(f))

print('All results in', RESULTS_DRIVE)